In [23]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

from utils import functions as func
from utils import parameters as params
from utils import metabolites as metab

from tqdm import tqdm
import copy
import pickle
import sympy

import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/other/test_lp/'

In [24]:
jabba = True
counter = 6
base = 5 
mu_val = 1e-9
fn = '/data2/hratch/human_me/other/test_lp/S_matrix.h5'

In [4]:
# from expression import build_me_model
# tme, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = True, 
#                                             dummy_protein = False)

# if jabba:
#     for r in tme.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
    
    
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

# with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [25]:
with open(lp_path + 'working_version_' + str(base) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme1 = pickle.load(handle)
    


In [ ]:
sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)    
sln1, stat1, _ = tme1.solve_lp(mu_val = mu_val)    

In [5]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme0.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln0[tme0.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,4.899296e-10
protein_biomass_to_biomass,protein_biomass_to_biomass,3.280437e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
DNA_biomass_to_biomass,DNA_biomass_to_biomass,1.400000e-11


In [ ]:
# with open(lp_path + 'working_version' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [ ]:
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

S = tme.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(tme.reactions)]})
res.index = [r.id for r in tme.reactions]

if stat == 0:
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)
    print('Model did not solve')
    
res.loc[[i for i in res.index if 'biomass' in i],:].sort_values(by = 'reaction_fluxes', ascending = False)

In [4]:
def save_me_model(me_model, counter):
    print('Success, please update git')
#     lp_path = '/data2/hratch/human_me/test_lp/'
#     me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')

def get_changes(S_1, S_0):
    mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
    am = S_1.index.tolist()
    mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm:
        mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

    am, rm = S_1.index.tolist(), S_1.columns.tolist()
    mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm_2:
        mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))
    
    return mismatch, mm, mm_2

In [26]:
# S_0 = pd.read_hdf(fn, key = str(base))
# S_1 = pd.read_hdf(fn, key = str(counter))
S_1 = tme1.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
S_0 = tme0.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')

In [5]:
# rows = list(set(S_0.index).intersection(S_1.index))
# columns = list(set(S_0.columns).intersection(S_1.columns))

# s0 = S_0.loc[rows, columns].values
# s1 = S_1.loc[rows, columns].values

# mismatch = np.argwhere(np.not_equal(s0, s1))

# am = S_1.index.tolist()
# mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
# for m in mm:
#     mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

# am, rm = S_1.index.tolist(), S_1.columns.tolist()
# mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
# for m in mm_2:
#     mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))





In [128]:
# mapper = {
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_COPI_RETROtr': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_COPI_RETROtr',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_DEUBIQUITINATIONc': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_DEUBIQUITINATIONc',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_FORMATIONg': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_FORMATIONg', 
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_POLYUBIQUITINATIONc': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_POLYUBIQUITINATIONc',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_PROTEASOMAL_DEGRADATIONc': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_PROTEASOMAL_DEGRADATIONc',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_RETROTRANSLOCATION': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_RETROTRANSLOCATION',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_UNFOLDr': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_UNFOLDr'
   
# }

# S_1.columns = [col if col not in mapper else mapper[col] for col in S_1.columns.tolist()]

# # mapper = {
# #     '266375715_complex_n': '125250296_complex_n', 
# #     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_enzyme_deg_proxy': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_enzyme_deg_proxy',
# #     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_c': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_c',
# #     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_g': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_g',
# #     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_r': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_r',
# #     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_polyub_complex_c': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_polyub_complex_c',
# #     'unfolded_Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_r': 'unfolded_Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_r'
# # }
# mapper = {'191091342_complex_n': '125250296_complex_n'}
# S_1.index = [col if col not in mapper else mapper[col] for col in S_1.index.tolist()]


# S_1 = S_1.loc[S_0.index.tolist(), S_0.columns.tolist()]

In [149]:
if S_1.shape != S_0.shape:
    print('Dimensions are not the same')
    indeces = False
if len(set(S_1.columns).difference(S_0.columns)) > 0:
    print('Columns are not the same')
    indeces = False
if len(set(S_1.index).difference(S_0.index)) > 0:
    print('Rows are not the same')
    indeces = False

In [15]:
S_1.shape != S_0.shape

False

In [27]:
# S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
            indeces = False
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch, mm, mm_2 = get_changes(S_1, S_0)
            print('Dataframes are not equal due to stoichiometric values mismatch, will not save model')
        else:
            print('Same')
#             save_me_model(tme, counter)
    else:
        print('Dataframes are not equal due to column/row label mismatch, will not save model')
else:
    print('Same')
#     save_me_model(tme, counter)

Same


In [ ]:
# idx = S_1.index.tolist()
# for i in range(len(idx)):
#     val = idx[i]
#     if 'folded' in val and 'polyub' in val:
#         idx[i] = val.replace('protein_polyub', 'protein_' + val[-1] + '_polyub')
# S_1.index = idx

# idx = set(S_0.index).intersection(S_1.index)
# col = set(S_0.columns).intersection(S_1.columns)
# S_0 = S_0.loc[idx, col]
# S_1 = S_1.loc[idx, col]

print(set(S_1.index).difference(S_0.index))
print(set(S_0.index).difference(S_1.index))

for tr, v in dict(zip(list(set(S_1.index).difference(S_0.index)), list(set(S_0.index).difference(S_1.index)))).items():
    S_1.index = pd.Series(S_1.index).replace(to_replace = tr, value = v, inplace = False)

In [25]:
# S_1 = S_1.loc[rows,columns]
# S_0 = S_0.loc[rows,columns]
# mismatch, mm, mm_2 = get_changes(S_1, S_0)

In [7]:
list(mm.keys())

[15, 42, 49, 1152, 1153, 1413]

In [9]:
m_idx = 15
mm[m_idx]

{'id': 'amp_c', 'reactions': [5376, 5401, 5509, 5699, 5709, 5872, 6998]}

In [10]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:5234_DECAPPING_mRNA_DEGRADATIONc      832.0
HGNC:24432_DECAPPING_mRNA_DEGRADATIONc     404.0
HGNC:11985_DECAPPING_mRNA_DEGRADATIONc    1338.0
HGNC:29200_DECAPPING_mRNA_DEGRADATIONc    1809.0
HGNC:24971_DECAPPING_mRNA_DEGRADATIONc     726.0
HGNC:15502_DECAPPING_mRNA_DEGRADATIONc     458.0
HGNC:3419_DECAPPING_mRNA_DEGRADATIONc     1697.0
Name: amp_c, dtype: float64

In [11]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:5234_DECAPPING_mRNA_DEGRADATIONc      842.0
HGNC:24432_DECAPPING_mRNA_DEGRADATIONc     398.0
HGNC:11985_DECAPPING_mRNA_DEGRADATIONc    1325.0
HGNC:29200_DECAPPING_mRNA_DEGRADATIONc    1855.0
HGNC:24971_DECAPPING_mRNA_DEGRADATIONc     680.0
HGNC:15502_DECAPPING_mRNA_DEGRADATIONc     432.0
HGNC:3419_DECAPPING_mRNA_DEGRADATIONc     1782.0
Name: amp_c, dtype: float64

In [18]:
r_id = 'HGNC:10425_DECAPPING_mRNA_DEGRADATIONc'

tme0.reactions.get_by_id(r_id).reaction

'1.1104132525694678e-07 DECAPPING_mRNA_DEGRADATIONc_COMPLEX_enzyme_degradation_proxy_c + 6.053762505166e6*mu  1.11041325256947e7 DECAPPING_mRNA_DEGRADATIONc_complex_c + HGNC:10425_mrna_c + 2 ahcys_c + 446.007883287 biomass_mRNA + 1385 h2o_c --> HGNC:10425_mrna_degradation_proxy_c + 2 amet_c + 513 amp_c + 251 cmp_c + gdp_c + 290 gmp_c + 1385 h_c + 331 ump_c'

In [19]:
tme1.reactions.get_by_id(r_id).reaction

'1.1104132525694678e-07 DECAPPING_mRNA_DEGRADATIONc_COMPLEX_enzyme_degradation_proxy_c + 6.053762505166e6*mu  1.11041325256947e7 DECAPPING_mRNA_DEGRADATIONc_complex_c + HGNC:10425_mrna_c + 2 ahcys_c + 417.12645919899995 biomass_mRNA + 1297 h2o_c --> HGNC:10425_mrna_degradation_proxy_c + 2 amet_c + 425 amp_c + 251 cmp_c + gdp_c + 290 gmp_c + 1297 h_c + 331 ump_c'

In [14]:
hgnc_ids = set([m.split('_')[0] for m in S_0.iloc[m_idx, mm[m_idx]['reactions']].index])

In [15]:
hgnc_ids

{'HGNC:11985',
 'HGNC:15502',
 'HGNC:24432',
 'HGNC:24971',
 'HGNC:29200',
 'HGNC:3419',
 'HGNC:5234'}

In [15]:
from utils import machinery as mach
hgnc_ids.intersection(mach.rs['HGNC ID (gene)'])

set()

In [16]:
hgnc_ids.intersection(mach.rl['HGNC ID (gene)'])

set()

In [17]:
hgnc_ids.difference(mach.rl['HGNC ID (gene)'].tolist() + mach.rs['HGNC ID (gene)'].tolist())

{'HGNC:10073', 'HGNC:33853'}

In [13]:
tme0.expressed_genes['HGNC:5234'].reactions

{'Catalysis_Reactions': {'Metabolic_Module': {},
  'Expression_Module': {'HGNC:10683_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzyme_degradation'},
   'HGNC:30242_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzyme_degradation'},
   'HGNC:830_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzyme_degradation'},
   'HGNC:7697_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzyme_degradation'},
   'HGNC:22921_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzyme_degradation'},
   'HGNC:2898_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzyme_degradation'},
   'HGNC:4803_IMPORTtm': {'IMPORTtm_complex_m': 'catalysis',
    'IMPORTtm_COMPLEX_enzyme_degradation_proxy_m': 'enzym

In [26]:
tme0.metabolites.get_by_id('HGNC:10073_mrna_c').sequence



Seq('GCAUUGUGGGUUCUCCUGGAGCUGUGGAGUUGAUCCUGAAUGAAAGUGGCGCGC...AAA', RNAAlphabet())

In [27]:
len(tme0.metabolites.get_by_id('HGNC:10073_mrna_c').sequence)

4936

In [29]:
tme0.metabolites.get_by_id('HGNC:10073_mrna_c').sequence == tme1.metabolites.get_by_id('HGNC:10073_mrna_c').sequence[:len(tme0.metabolites.get_by_id('HGNC:10073_mrna_c').sequence)]

True

In [30]:
tme1.expressed_genes['HGNC:33853'].macromolecules

{'RNA': {'premrna': None,
  'mrna': {'coupled': {'HGNC:33853_mrna_c': ['HGNC:33853_TRANSLATION_ELONGATIONc']},
   'other': None},
  'lariat': 'HGNC:33853_lariat_n'},
 'Protein': {'coupled': {},
  'other': ['HGNC:33853_unfolded_protein_c',
   'HGNC:33853_folded_protein_c',
   'HGNC:33853_folded_protein_n'],
  'non-machinery': []},
 'Complex': {'coupled': {'TRANSCRIPTION_complex_n': ['HGNC:11551_TRANSCRIPTION',
    'HGNC:28199_TRANSCRIPTION',
    'HGNC:17315_TRANSCRIPTION',
    'HGNC:7685_TRANSCRIPTION',
    'HGNC:2269_TRANSCRIPTION',
    'HGNC:18170_TRANSCRIPTION',
    'HGNC:7758_TRANSCRIPTION',
    'HGNC:30858_TRANSCRIPTION',
    'HGNC:10425_TRANSCRIPTION',
    'HGNC:28883_TRANSCRIPTION',
    'HGNC:24932_TRANSCRIPTION',
    'HGNC:20947_TRANSCRIPTION',
    'HGNC:3133_TRANSCRIPTION',
    'HGNC:11054_TRANSCRIPTION',
    'HGNC:652_TRANSCRIPTION',
    'HGNC:30758_TRANSCRIPTION',
    'HGNC:4458_TRANSCRIPTION',
    'HGNC:2325_TRANSCRIPTION',
    'HGNC:4341_TRANSCRIPTION',
    'HGNC:381_TRANSC

In [21]:
tme1.expressed_genes['HGNC:33853'].reactions

{'Catalysis_Reactions': {'Metabolic_Module': {},
  'Expression_Module': {'HGNC:9141_TRANSCRIPTION': {'TRANSCRIPTION_complex_n': 'catalysis',
    'TRANSCRIPTION_COMPLEX_enzyme_degradation_proxy_n': 'enzyme_degradation'},
   'HGNC:10369_TRANSCRIPTION': {'TRANSCRIPTION_complex_n': 'catalysis',
    'TRANSCRIPTION_COMPLEX_enzyme_degradation_proxy_n': 'enzyme_degradation'},
   'HGNC:11441_TRANSCRIPTION': {'TRANSCRIPTION_complex_n': 'catalysis',
    'TRANSCRIPTION_COMPLEX_enzyme_degradation_proxy_n': 'enzyme_degradation'},
   'HGNC:15887_TRANSCRIPTION': {'TRANSCRIPTION_complex_n': 'catalysis',
    'TRANSCRIPTION_COMPLEX_enzyme_degradation_proxy_n': 'enzyme_degradation'},
   'HGNC:17351_TRANSCRIPTION': {'TRANSCRIPTION_complex_n': 'catalysis',
    'TRANSCRIPTION_COMPLEX_enzyme_degradation_proxy_n': 'enzyme_degradation'},
   'HGNC:4664_TRANSCRIPTION': {'TRANSCRIPTION_complex_n': 'catalysis',
    'TRANSCRIPTION_COMPLEX_enzyme_degradation_proxy_n': 'enzyme_degradation'},
   'HGNC:24661_TRANSCRIPTI

In [24]:
m1 = tme1.metabolites.get_by_id('HGNC:10425_mrna_c')

In [26]:
m0.sequence

Seq('CUCUUCCGUCGCAGAGUUUCGCCAUGGCCCGGGGCCCCAAGAAGCACUUAAAGC...AAA', RNAAlphabet())

In [27]:
m1.sequence

Seq('CUCUUCCGUCGCAGAGUUUCGCCAUGGCCCGGGGCCCCAAGAAGCACUUAAAGC...AAA', RNAAlphabet())

In [31]:
m0.sequence[:len(m1.sequence)] == m1.sequence

True

1297

In [16]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-12]

Series([], Name: biomass_protein, dtype: float64)

In [290]:
tme0.reactions.get_by_id('HGNC:12463_MONOMERIZATIONc').reaction

'HGNC:12463_unfolded_protein_c + 8.972431523266685e-07 HGNC:12628_enzyme_deg_proxy + 6.07704347928105e5*mu  8.97243152326669e7 HGNC:12628_folded_protein_c + 0.0671123399999907 biomass_protein + 3 h2o_c --> cys_L_c + 3 ubiquitin_monomer_protein_c'

In [291]:
tme1.reactions.get_by_id('HGNC:12463_MONOMERIZATIONc').reaction

'HGNC:12463_unfolded_protein_c + 6.07704347928105e5*mu  8.97243152326669e7 HGNC:12628_folded_protein_c + 0.0671123399999907 biomass_protein + 3 h2o_c --> cys_L_c + 3 ubiquitin_monomer_protein_c'

In [ ]:
# res0 = pd.read_csv(lp_path + 'works_trash.csv', index_col = 0)
# res['og_fluxes'] = res0.loc[res.index.tolist(), :]['reaction_fluxes'].tolist()
# res['diff'] = res['reaction_fluxes'] - res['og_fluxes']

# testing ubiquitin cleavage

In [ ]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    
